In [ ]:
import chromadb
import torch
from chromadb.api.types import Images

In [ ]:
# 1. 创建客户端
client = chromadb.Client()

In [ ]:
# 2. 创建集合
collection = client.create_collection(name="my_collection")

## 默认嵌入函数

In [ ]:
print(collection.configuration.get("embedding_function"))

## 自定义嵌入函数

In [ ]:
import numpy as np
from chromadb import EmbeddingFunction, Documents, Embeddings


class MyEmbeddingFunction(EmbeddingFunction[Documents]):
    # 初始化
    def __init__(self, len) -> None:
        self.len = len
        return

    # 调用方法
    def __call__(self, input: Documents) -> Embeddings:
        # 返回长度为 len 的数组
        return [np.full(self.len, len(item)) for item in input]


In [ ]:
# 创建集合指定嵌入函数
collection2 = client.get_or_create_collection(name="my_collection2", embedding_function=MyEmbeddingFunction(10))

In [ ]:
# 插入数据
collection.add(
    ids=["1", "2", "3"],
    documents=["hello world", "foo bar", "lorem ipsum"],
)

In [ ]:
collection2.peek()

In [ ]:
# 真实场景处理的嵌入函数
class ImageEmbeddingFunction(EmbeddingFunction[Images]):
    # 传入自己的嵌入模型
    def __init__(self, model) -> None:
        self.model = model.to('cpu')
        return

    def __call__(self, input: Images) -> Embeddings:
        # 将输入图像转换为tensor
        input_tensor = torch.tensor(np.array(input))
        # 前向传播
        with torch.no_grad():
            embeddings = self.model(input_tensor)
        # 转成ndarray返回
        return embeddings.numpy()